MD# Degrau 1: Gemini via API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/souuzaa/gdg-blumenau-ml-gcp/blob/main/03-gemini-api/gemini_penguins.ipynb)

Ninguém treina nada: mandamos as medidas de 20 pinguins para o Gemini e pedimos a espécie. Depois comparamos com a regressão logística do caminho 1.

1. Gere uma chave grátis em [aistudio.google.com/apikey](https://aistudio.google.com/apikey).
2. No Colab, abra **Secrets** (ícone de chave na barra lateral) e crie `GEMINI_API_KEY`.

In [ ]:
%pip install -q google-genai

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass  # fora do Colab: exporte GEMINI_API_KEY no terminal

from google import genai
from google.genai import types

client = genai.Client()
MODELO = "gemini-flash-latest"  # se der erro, rode: [m.name for m in client.models.list()]

In [ ]:
import pandas as pd
from pathlib import Path

CSV_URL = "https://raw.githubusercontent.com/souuzaa/gdg-blumenau-ml-gcp/main/data/penguins.csv"
local = Path("../data/penguins.csv")
df = pd.read_csv(local if local.exists() else CSV_URL)

# Igual ao slide: remove qualquer linha com valor vazio.
df = df.dropna()
print(len(df), "pinguins")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
ESPECIES = ["Adelie", "Chinstrap", "Gentoo"]
treino, teste = train_test_split(df, test_size=0.2, random_state=42)  # mesmo split do caminho 1
teste = teste.head(20)
EXEMPLOS_POR_ESPECIE = 0  # @param {type:"integer"}
exemplos = treino.groupby("species").head(EXEMPLOS_POR_ESPECIE) if EXEMPLOS_POR_ESPECIE else treino.head(0)

def descrever(l):
    return (f"bico {l.bill_length_mm} mm de comprimento e {l.bill_depth_mm} mm de altura, "
            f"nadadeira {l.flipper_length_mm} mm, massa {l.body_mass_g} g")

linhas = ["Você é biólogo e conhece os pinguins do arquipélago Palmer, na Antártida.",
          "Classifique cada pinguim como Adelie, Chinstrap ou Gentoo, só pelas medidas.",
          f"Devolva uma lista JSON com exatamente {len(teste)} espécies, na mesma ordem."]
if len(exemplos):
    linhas += ["", "Exemplos com a resposta certa:"] + [f"- {descrever(l)}: {l.species}" for l in exemplos.itertuples()]
linhas += ["", "Pinguins para classificar:"] + [f"{i + 1}. {descrever(l)}" for i, l in enumerate(teste.itertuples())]
prompt = "\n".join(linhas)
print(prompt[:800])

In [ ]:
import json

resposta = client.models.generate_content(
    model=MODELO,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json",
        response_schema={"type": "ARRAY", "items": {"type": "STRING", "enum": ESPECIES}},
    ),
)
previsto_llm = json.loads(resposta.text)
print(resposta.usage_metadata)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

classico = make_pipeline(StandardScaler(), LogisticRegression()).fit(treino[FEATURES], treino["species"])
comparacao = pd.DataFrame({
    "real": teste["species"].values,
    "gemini": (previsto_llm + ["?"] * len(teste))[: len(teste)],
    "regressao_logistica": classico.predict(teste[FEATURES]),
})
print("Gemini (sem treino):           ", (comparacao.real == comparacao.gemini).sum(), "/", len(teste))
print("Regressão logística (treinada):", (comparacao.real == comparacao.regressao_logistica).sum(), "/", len(teste))
comparacao

MD## O que levar daqui

Dado tabular com rótulo: um modelo pequeno e treinado costuma ganhar e custa quase nada. Texto, imagem e áudio sem rótulo: aí o degrau 1 brilha.

Experimente `EXEMPLOS_POR_ESPECIE = 3` (few-shot) e rode de novo.